In [ ]:
F1_THRESHOLD = 0.0

import json
import pandas as pd
import os

columns = ["focal", "depth", "filters", "kernel", "dropout", "dense_depth", "dense"]
experiments = ["Bin_SDSS", "Bin_DES", "Bin_H_1", "Bin_H_2", "Bin_H_3"]
setups = ["paper3_models"]#, "paper3_models_same_padding"]

dataset = pd.DataFrame({}, columns= columns + ["best_f1"])

for experiment in experiments:
    for setup in setups:
        for i in range(1000):
            path_base = f'{setup}/{experiment}/{i}'
            if not os.path.exists(path_base):
                continue
            with open(f'{path_base}/model.json', 'r') as mf:
                model_dict = json.load(mf)
            if os.path.exists(f'{path_base}/result.json'):
                with open(f'{path_base}/result.json', 'r') as rf:
                    res_dict = json.load(rf)
                model_dict["best_f1"] = res_dict["best_f1"]
            else:
                model_dict["best_f1"] = None
            model_dict["experiment"] = experiment
            model_dict["setup"] = setup
            data = pd.DataFrame(model_dict,  index = [i])
            dataset = pd.concat([dataset, data])

dataset = dataset[~dataset["best_f1"].isna()]
dataset = dataset[dataset["best_f1"] >= F1_THRESHOLD]
dataset["relative_f1"] = dataset.apply(lambda x: x["best_f1"] / dataset[dataset["experiment"] == x["experiment"]]["best_f1"].max(), axis=1)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import wilcoxon, ttest_ind

TOP_N_VALUES = 50


columns = ["focal", "depth", "filters", "kernel", "dropout", "dense_depth", "dense"]

plt.rcParams.update({'font.size': 14})

for j, column in enumerate(columns):
    fig = plt.figure(figsize=(12, 6))
    gs = gridspec.GridSpec(2, 6, figure=fig)
    
    # Top row (2 centered plots)
    ax1 = fig.add_subplot(gs[0, 1:3])  # middle-left
    ax2 = fig.add_subplot(gs[0, 3:5])  # middle-right
    
    # Bottom row (3 evenly spaced plots)
    ax3 = fig.add_subplot(gs[1, 0:2])
    ax4 = fig.add_subplot(gs[1, 2:4])
    ax5 = fig.add_subplot(gs[1, 4:6])
    
    axs = [ax1, ax2, ax3, ax4, ax5]
    
    for i, experiment in enumerate(experiments):
        ex_data = dataset[dataset["experiment"] == experiment]
        
        values = []
        max_values = []
        top_n_focal = []
        top_n_non_focal = []
        unique_parameter_vales = sorted(ex_data[column].unique())

        for value in unique_parameter_vales:
            parameter_values = ex_data[ex_data[column] == value]
            
            values.append(parameter_values["best_f1"])
            max_values.append(parameter_values["best_f1"].max())

            focal_values = parameter_values[parameter_values["focal"] == True]
            non_focal_values = parameter_values[parameter_values["focal"] == False]
            
            top_focal_values = focal_values.sort_values("best_f1").tail(TOP_N_VALUES)["best_f1"]
            top_n_focal.append(top_focal_values)

            top_non_focal_values = non_focal_values.sort_values("best_f1").tail(TOP_N_VALUES)["best_f1"]
            top_n_non_focal.append(top_non_focal_values)
            
        positions = range(1, len(unique_parameter_vales) + 1)
        
        bp_focal = axs[i].boxplot(top_n_focal, positions = positions, labels = unique_parameter_vales, showfliers=False, patch_artist=True, boxprops=dict(facecolor='green', alpha = 0.5))
        bp_non_focal = axs[i].boxplot(top_n_non_focal, positions = positions, labels = unique_parameter_vales, showfliers=False, patch_artist=True, boxprops=dict(facecolor='red', alpha = 0.5))

        for x in range(len(top_n_focal)):
            if len(top_n_focal[x]) != len(top_n_non_focal[x]):
                continue
            
            whisker_focal = bp_focal['whiskers'][2*x + 1]  # top whisker
            whisker_non_focal = bp_non_focal['whiskers'][2*x + 1]  # top whisker
            
            y_focal = max(whisker_focal.get_ydata())
            y_non_focal = max(whisker_non_focal.get_ydata())
            y_coord = max(y_focal, y_non_focal)
            
            res = ttest_ind(top_n_focal[x], top_n_non_focal[x])
            p_val = res.pvalue
            axs[i].text(x + 1, y_coord , f"{p_val:.3f}", ha='center', size = 10, color = "green" if p_val < 0.05 else "red")
            
        # axs[i].plot(positions, max_values, color="red", label="Max F1 value", marker='o')
        # axs[i].plot(positions, top_5_max_values, color="royalblue", label="Top 10 F1 values, avg", marker='o')
        # axs[i].legend()
        axs[i].set_title(experiment)
        axs[i].set_ylabel("F1")
        axs[i].grid()
        # plt.show()
    # plt.legend()

    # Get handles/labels from ONE axis
    handles, labels = ax1.get_legend_handles_labels()
    # Global legend
    fig.legend(handles, labels, loc='upper left', ncol=2)
    fig.suptitle(column.title())
    plt.tight_layout()
    # plt.savefig(f"paper3_figures/step_1_{column}.pdf")
    
    
    # ax[j][i].boxplot(values, positions = positions, labels = unique_parameter_vales, showfliers=False)

# for column in ["depth", "filters", "kernel", "dropout", "dense_depth", "dense"]:




In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import wilcoxon, ttest_ind

TOP_N_VALUES = 20


columns = ["focal", "depth", "filters", "kernel", "dropout", "dense_depth", "dense"]

column_names = {
    "focal": "Focal",
    "depth": "Number of Convolution Blocks",
    "filters": "Convolution Channels",
    "kernel": "Convolution Kernel Size",
    "dropout": "Dropout",
    "dense_depth": "Number of Dense Layers",
    "dense": "Dense Outputs"
}

bin_names = {
    "Bin_SDSS": "SDSS",
    "Bin_DES": "DECam",
    "Bin_H_1": "Hubble 1",
    "Bin_H_2": "Hubble 2",
    "Bin_H_3": "Hubble 3",
}

plt.rcParams.update({'font.size': 14})

for j, column in enumerate(columns):
    fig = plt.figure(figsize=(12, 6), constrained_layout=True)
    gs = gridspec.GridSpec(2, 6, figure=fig)
    
    # Top row (2 centered plots)
    ax1 = fig.add_subplot(gs[0, 1:3])  # middle-left
    ax2 = fig.add_subplot(gs[0, 3:5])  # middle-right
    
    # Bottom row (3 evenly spaced plots)
    ax3 = fig.add_subplot(gs[1, 0:2])
    ax4 = fig.add_subplot(gs[1, 2:4])
    ax5 = fig.add_subplot(gs[1, 4:6])
    
    axs = [ax1, ax2, ax3, ax4, ax5]
    
    for i, experiment in enumerate(experiments):
        ex_data = dataset[dataset["experiment"] == experiment]
        
        values = []
        max_values = []
        top_n_values = []
        unique_parameter_vales = sorted(ex_data[column].unique())

        for value in unique_parameter_vales:
            parameter_values = ex_data[ex_data[column] == value]
            
            values.append(parameter_values["best_f1"])
            max_values.append(parameter_values["best_f1"].max())

            
            top_values = parameter_values.sort_values("best_f1").tail(TOP_N_VALUES)["best_f1"]
            top_n_values.append(top_values)
            
        positions = range(1, len(unique_parameter_vales) + 1)
        
        axs[i].boxplot(top_n_values, positions = positions, labels = unique_parameter_vales, showfliers=False)
            
        axs[i].plot(positions, max_values, color="red", label="Max F1 value", marker='o')
        # axs[i].plot(positions, top_n_values, color="royalblue", label="Top 10 F1 values, avg", marker='o')
        if i == 0:
            axs[i].legend()
        axs[i].set_title(bin_names[experiment])
        axs[i].set_ylabel("F1")
        axs[i].grid()
    # plt.tight_layout()
    fig.suptitle(column_names[column].title(), fontweight="bold")
    plt.savefig(f"paper3_step1_{column}.pdf")
    # plt.legend()

    # Get handles/labels from ONE axis
    # handles, labels = ax1.get_legend_handles_labels()
    # Global legend
    # fig.legend(handles, labels, loc='upper left', ncol=2)
    
    
    # plt.savefig(f"paper3_figures/step_1_{column}.pdf")
    
    
    # ax[j][i].boxplot(values, positions = positions, labels = unique_parameter_vales, showfliers=False)

# for column in ["depth", "filters", "kernel", "dropout", "dense_depth", "dense"]:




In [ ]:
dataset[(dataset["experiment"] == "Bin_SDSS") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)

In [ ]:
dataset[(dataset["experiment"] == "Bin_SDSS") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)["dense"].value_counts()

In [ ]:
dataset[(dataset["experiment"] == "Bin_DES") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)

In [ ]:
dataset[(dataset["experiment"] == "Bin_DES") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)["dense"].value_counts()

In [ ]:
dataset[(dataset["experiment"] == "Bin_H_1") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)

In [ ]:
dataset[(dataset["experiment"] == "Bin_H_1") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)["dense"].value_counts()

In [ ]:
dataset[(dataset["experiment"] == "Bin_H_2") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)

In [ ]:
dataset[(dataset["experiment"] == "Bin_H_2") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)["dense"].value_counts()

In [ ]:
dataset[(dataset["experiment"] == "Bin_H_3") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)

In [ ]:
dataset[(dataset["experiment"] == "Bin_H_3") & (dataset["focal"] == False)].sort_values("best_f1").tail(10)["dense"].value_counts()